# Base64 인코딩 테스트
- `AI/data/images/product.jpg` → base64
- `AI/data/audio/scene_01.mp3` → base64

In [1]:
import base64
from pathlib import Path

DATA_DIR = Path("../data")

IMAGE_PATH = DATA_DIR / "images" / "product.jpg"
AUDIO_PATH = DATA_DIR / "audio" / "scene_01.mp3"

In [2]:
# 이미지 → base64
image_bytes = IMAGE_PATH.read_bytes()
image_b64 = base64.b64encode(image_bytes).decode("utf-8")

print(f"이미지 파일 크기: {len(image_bytes):,} bytes")
print(f"base64 문자열 길이: {len(image_b64):,} chars")
print(f"미리보기: {image_b64[:80]}...")

이미지 파일 크기: 275,805 bytes
base64 문자열 길이: 367,740 chars
미리보기: /9j/4AAQSkZJRgABAQAASABIAAD/2wBDAAEBAQEBAQEBAQEBAQEBAQEBAQEBAQEBAQEBAQEBAQEBAQEB...


In [3]:
# 오디오 → base64
audio_bytes = AUDIO_PATH.read_bytes()
audio_b64 = base64.b64encode(audio_bytes).decode("utf-8")

print(f"오디오 파일 크기: {len(audio_bytes):,} bytes")
print(f"base64 문자열 길이: {len(audio_b64):,} chars")
print(f"미리보기: {audio_b64[:80]}...")

오디오 파일 크기: 124,906 bytes
base64 문자열 길이: 166,544 chars
미리보기: SUQzBAAAAAECBVRTU0UAAAAOAAADTGF2ZjYwLjE2LjEwMEdFT0IAAQFjAAADYXBwbGljYXRpb24veC1j...


In [4]:
import json

# 워크플로우 JSON 로드
WORKFLOW_PATH = DATA_DIR / "comfyui" / "workflows" / "ltx2_workflow_v1.json"
workflow = json.loads(WORKFLOW_PATH.read_text(encoding="utf-8"))

# 노드 98(LoadImage)의 image → 업로드할 파일명으로 교체
workflow["98"]["inputs"]["image"] = IMAGE_PATH.name

# 노드 169(VHS_LoadAudioUpload)의 audio → 업로드할 파일명으로 교체
workflow["169"]["inputs"]["audio"] = AUDIO_PATH.name

# RunPod 페이로드 생성
runpod_payload = {
    "input": {
        "workflow": workflow,
        "images": [
            {
                "name": IMAGE_PATH.name,
                "image": image_b64,
            },
            {
                "name": AUDIO_PATH.name,
                "image": audio_b64,
            },
        ],
    }
}

print("RunPod payload 구조:")
print(f"  workflow 노드 수: {len(workflow)}")
print(f"  images 배열 길이: {len(runpod_payload['input']['images'])}")
print(f"  - {IMAGE_PATH.name}: {len(image_b64):,} chars")
print(f"  - {AUDIO_PATH.name}: {len(audio_b64):,} chars")

RunPod payload 구조:
  workflow 노드 수: 32
  images 배열 길이: 2
  - product.jpg: 367,740 chars
  - scene_01.mp3: 166,544 chars


In [5]:
# JSON 파일로 저장 (curl / Postman에서 바로 사용 가능)
OUTPUT_JSON = DATA_DIR / "comfyui" / "runpod_test_payload.json"
OUTPUT_JSON.parent.mkdir(parents=True, exist_ok=True)
OUTPUT_JSON.write_text(json.dumps(runpod_payload, indent=2, ensure_ascii=False), encoding="utf-8")

print(f"저장 완료: {OUTPUT_JSON}")
print(f"파일 크기: {OUTPUT_JSON.stat().st_size:,} bytes")
print()
print("테스트 명령어:")
print(f'curl -X POST "https://api.runpod.ai/v2/YOUR_ENDPOINT_ID/runsync" \\')
print(f'  -H "Authorization: Bearer YOUR_API_KEY" \\')
print(f'  -H "Content-Type: application/json" \\')
print(f'  -d @{OUTPUT_JSON}')

저장 완료: ..\data\comfyui\runpod_test_payload.json
파일 크기: 545,093 bytes

테스트 명령어:
curl -X POST "https://api.runpod.ai/v2/YOUR_ENDPOINT_ID/runsync" \
  -H "Authorization: Bearer YOUR_API_KEY" \
  -H "Content-Type: application/json" \
  -d @..\data\comfyui\runpod_test_payload.json


---
# RunPod 응답 디코딩
`AI/data/comfyui/serverless_response.txt` → MP4 파일로 저장

In [3]:
import json

DATA_DIR = Path("../data")

# RunPod 응답에서 MP4 디코딩
RESPONSE_PATH = DATA_DIR / "comfyui" / "serverless_response.txt"
response = json.loads(RESPONSE_PATH.read_text(encoding="utf-8"))

print(f"status: {response['status']}")

video_b64 = response["output"]["images"][0]["data"]
video_bytes = base64.b64decode(video_b64)

OUTPUT_DIR = DATA_DIR / "video"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
output_mp4 = OUTPUT_DIR / "serverless_result.mp4"
output_mp4.write_bytes(video_bytes)

print(f"디코딩 완료: {output_mp4}")
print(f"파일 크기: {len(video_bytes):,} bytes ({len(video_bytes)/1024/1024:.1f} MB)")

status: COMPLETED
디코딩 완료: ..\data\video\serverless_result.mp4
파일 크기: 1,376,783 bytes (1.3 MB)
